In [208]:
import time
import random
import json
import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types
from google.genai import errors
from dataclasses import dataclass, field

In [209]:
load_dotenv()
client = genai.Client()

### Gemini api call

In [210]:
@dataclass
class API_Configs:
    model_config = types.GenerateContentConfig(
    system_instruction="Respond STRICTLY with JSON format. Do not add markdown fences.",
    max_output_tokens=8192,
)
    
    model_tiers = {
    "high": {"model_name": "gemini-3.5-flash"},
    "medium": {"model_name": "gemini-2.5-flash"},
    "low": {"model_name": "gemini-3.1-flash-lite"}
}

In [211]:
@dataclass
class Retry_Config:
    max_attempts: int = 3
    base_delay: float = 1.0
    max_delay: float = 30.0

    retry_status_codes = (429, 500, 503, 504)

In [212]:
class Call_Mode:
    stream_request = "stream_request"
    embed_request = "embed_request"

In [213]:
def _handle_retry_or_raise(e, attempt, model_name):
    # Refers to the Tuple of status codes that can be retried in Retry_Config.
            if e.code in Retry_Config.retry_status_codes:
                if attempt == Retry_Config.max_attempts:
                    print("Max attempts reached.")
                    raise e

                # Exponential backoff, i should probably add a way to check the server header for the exact time needed if it exists later (keyword: *later*)
                new_delay = random.uniform(0.5, 1.5) * Retry_Config.base_delay * (2 ** (attempt - 1))
                clamped_delay = min(new_delay, Retry_Config.max_delay)

                print(f" Code {e.code}. Backing off. Waiting {clamped_delay:.2f}s...")
                time.sleep(clamped_delay)
                return

            else:
                status_reason = getattr(e, "status", "UNKNOWN")
                
                if e.code == 404 or status_reason == "NOT_FOUND": # Where is this error? i cant find it? Muhehehehehehhe
                    print(f"Model Not Found [404]: '{model_name}' does not exist or is deprecated.")
                    
                elif e.code in (401, 403) or status_reason == "PERMISSION_DENIED":
                    print(f"Authentication Failed [{e.code}]: Check your API key, project quotas, or permissions.")
                    
                elif e.code == 400 or status_reason == "INVALID_ARGUMENT":
                    print(f"Invalid Argument [400]: Malformed structure or prompt content limits exceeded.")
                    
                else:
                    print(f"Non-retryable error [{status_reason}] ({e.code}): {e.message}")
                    
                raise e

In [214]:
def _call_stream(model_name, prompt, config):
    for attempt in range(1, Retry_Config.max_attempts + 1):
        try:
            if attempt > 1:
                print(f"Retry attempt {attempt}/{Retry_Config.max_attempts}...")
            else:
                print("Sending initial request...")

            response = client.models.generate_content_stream(
                model=model_name, contents=prompt, config=config
            )
            for chunk in response:
                yield chunk
            return
        except errors.APIError as e:
            _handle_retry_or_raise(e, attempt, model_name)
            continue

In [215]:
def _call_embedding(model_name, prompt, config):
    for attempt in range(1, Retry_Config.max_attempts + 1):
        try:
            if attempt > 1:
                print(f"Retry attempt {attempt}/{Retry_Config.max_attempts}...")
            else:
                print("Sending initial request...")

            return client.models.embed_content(
                model=model_name, contents=prompt
            )
        except errors.APIError as e:
            _handle_retry_or_raise(e, attempt, model_name)
            continue

In [216]:
def execute_call(model_priority, prompt, config, mode):

    tier = API_Configs.model_tiers.get(model_priority.lower(), API_Configs.model_tiers["medium"])
    

    mode = mode
    for attempt in range (1, Retry_Config.max_attempts + 1):

        # Api call Section
        if mode == Call_Mode.stream_request:

            model_name = tier["model_name"]
            return _call_stream(model_name , prompt, config)
                        
        elif mode == Call_Mode.embed_request:
            
            model_name = "gemini-embedding-2"
            return _call_embedding(model_name, prompt, config)
            

In [217]:
def to_rag_records(result, source_texts, source_ids=None):
    records = []
    for i, emb in enumerate(result.embeddings):
        records.append({
            "id": source_ids[i] if source_ids else i,
            "text": source_texts[i],
            "vector": np.array(emb.values, dtype=np.float32),
            "dim": len(emb.values),
        })
    return records

In [218]:
def get_structured_output(model_priority, prompt, config, mode):
    response = execute_call(model_priority=model_priority, prompt=prompt, config=config, mode=mode)

    if mode == Call_Mode.stream_request:
        full_response = "".join(getattr(chunk, "text", "") or "" for chunk in response)
        data = json.loads(full_response)

    elif mode == Call_Mode.embed_request:
        data = to_rag_records(response, source_texts=[prompt])

    else:
        raise ValueError(f"Unsupported mode: {mode}")

    return data

In [219]:
#prompt = "You shall henceforth and at once provide me with TEN of the greatest most glorified spaghettie recipes that have crossed your existence."
#prompt = "Say hello in 10 random languages and state which language it is"

In [220]:
#result = get_structured_output(model_priority = "high", prompt = prompt, config = API_Configs.model_config, mode = Call_Mode.stream_request)

In [221]:
#print(result)

In [ ]:
# === INTERACTIVE TERMINAL SIMULATOR CELL (WITH UI FLUSH FIX) ===
import sys
import time

print("====================================================")
print("  🗣️  Gemini Interactive Chat Terminal Simulator 🗣️")
print("  Type 'exit' or 'quit' to end the session.")
print("====================================================")
sys.stdout.flush() # Force initial render

while True:
    try:
        # 1. Grab user input
        user_prompt = input("\nEnter your prompt (e.g., Ask for recipes, translations, etc.):\n> ")
        
        if user_prompt.strip().lower() in ['exit', 'quit']:
            print("\n👋 Terminal session closed. Happy coding!")
            sys.stdout.flush()
            break
            
        if not user_prompt.strip():
            continue

        print("\nThinking... Calling API...")
        sys.stdout.flush() # Force "Thinking..." to show up immediately

        # 2. Leverage your existing variables and functions completely untouched
        result_data = get_structured_output(model_priority = "high", prompt=user_prompt, config = API_Configs.model_config, mode = Call_Mode.stream_request)
        
        # 3. Print the formatted structured output
        print("\n" + "="*50)
        print("📥 RESPONSE:")
        print("="*50)
        print(result_data)
        print("="*50)
        
        # 4. CRITICAL: Force Jupyter to flush the text stream to the screen
        sys.stdout.flush()
        
        # Give the Jupyter notebook front-end a 50ms breather to render the text
        time.sleep(0.05)

    except json.JSONDecodeError:
        print("\n❌ Parsing Error: The model returned unstructured text or invalid JSON.")
        sys.stdout.flush()
    except Exception as e:
        print(f"\n❌ Error occurred: {e}")
        sys.stdout.flush()

  🗣️  Gemini Interactive Chat Terminal Simulator 🗣️
  Type 'exit' or 'quit' to end the session.

Thinking... Calling API...
Sending initial request...

📥 RESPONSE:
[{'id': 0, 'text': 'She pressed the knife further into his neck', 'vector': array([ 0.02354472, -0.00724052,  0.00766712, ..., -0.0015462 ,
        0.00126448,  0.00348671], dtype=float32), 'dim': 3072}]

👋 Terminal session closed. Happy coding!
